# Sesión 10 · MCP: herramientas conectadas

**AI Engineer I: Fundamentos de IA Generativa** · Módulo 1 · Edutek

---

## 🎯 Objetivos

Al terminar este notebook podrás:

1. **Explicar qué es MCP** y qué problema resuelve (el problema N×M): arquitectura *host / cliente / servidor*, transportes y ciclo de vida.
2. **Usar las tres primitivas** con ejemplos ejecutables: *tools* (las decide el modelo), *resources* (los carga la aplicación) y *prompts* (los elige la persona).
3. **Explorar un servidor existente** con un cliente mínimo y leer, mensaje a mensaje, lo que se dicen cliente y servidor.
4. **Construir un servidor MCP propio** con `FastMCP` (SDK oficial de Python) que **lee y escribe** en un CRM.
5. **Conectar un agente de LangChain** con `langchain-mcp-adapters`: a tu servidor, con aprobación humana antes de escribir, y a **dos servidores a la vez**.
6. **Registrar tu servidor en un cliente MCP real** (Claude Code, Claude Desktop) y **depurarlo** (logs a `stderr`, Inspector, cliente mínimo).
7. **Servir tu agente como una API** (FastAPI) con **sesiones persistentes** (`session_id` = `thread_id` del checkpointer, en SQLite) y entender cómo lo usan una interfaz (Streamlit) y un cliente HTTP (Bruno).

## 🗺️ Mapa

| Parte | Tema | ¿Consume dinero? |
| --- | --- | --- |
| 0 | Preparar el entorno | No |
| 1 | El problema N×M: por qué existe MCP | No |
| 2 | Arquitectura, transportes y ciclo de vida (a mano, en JSON-RPC) | No |
| 3 | Las tres primitivas: tools, resources y prompts | No |
| 4 | Explorar un servidor existente | No (descarga un paquete pequeño) |
| 5 | Construir tu propio servidor MCP (un CRM) | No |
| 6 | El agente con `langchain-mcp-adapters` | No (modelo de juguete) |
| 7 | Dos servidores a la vez (multi-servidor) | No (modelo de juguete) |
| 8 | Registrar tu servidor en un cliente real | No |
| 9 | Depuración | No |
| 10 | Servir el agente: API, sesiones e interfaz | No (modelo de juguete) |
| 11 | La práctica: `langchain-mcp-advanced-tooling` | No en modo simulado |
| 12 | Cierre | — |

> 🧸 **Modelo de juguete y CRM simulado.** Para ver el *mecanismo* sin red ni costo, el notebook usa un modelo "guionado" que responde con decisiones preparadas y un CRM de mentira en memoria con **datos 100 % ficticios**. Un modelo real decide por sí mismo; el mecanismo que lo rodea (servidor MCP, herramientas, aprobación) es **idéntico**. La práctica final usa HubSpot real si tienes cuenta (plan gratuito) y el modo simulado si no.

> 📌 **Versiones.** Probado con `mcp 1.30.0` (SDK oficial de Python, protocolo `2025-11-25`), `langchain-mcp-adapters 0.3.2`, `langchain 1.5.0` y `langgraph 1.2.22` (noviembre de 2026). La especificación de MCP evoluciona deprisa: hay una revisión **`2026-07-28`** más nueva que la que implementa el SDK; las diferencias relevantes están marcadas en la Parte 2.

> ℹ️ **Alcance.** Esta sesión trata de **cómo funciona MCP y cómo construir con él**. La seguridad de MCP se verá en el Módulo 2.

---
# Parte 0 · Preparar el entorno

In [ ]:
import sys

print("Python:", sys.version.split()[0])
assert sys.version_info[:2] == (3, 12), (
    "Esta sesión está fijada a Python 3.12. Ejecuta `uv sync` dentro de la carpeta "
    "s10-mcp-herramientas y selecciona el kernel .venv (ver README)."
)

In [ ]:
import asyncio
import json
import logging
import shutil
import sys
import tempfile
import warnings
from importlib.metadata import version
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
warnings.filterwarnings("ignore")
logging.getLogger("mcp").setLevel(logging.WARNING)   # el SDK registra cada petición en INFO: aquí sería ruido
logging.getLogger("httpx").setLevel(logging.WARNING)

for paquete in ("mcp", "langchain-mcp-adapters", "langchain", "langgraph"):
    print(f"{paquete:<24}{version(paquete)}")

# Los servidores de ejemplo se escriben en una carpeta temporal: no ensucian el repositorio.
TRABAJO = Path(tempfile.mkdtemp(prefix="s10_"))
sys.path.insert(0, str(TRABAJO))
print("Carpeta de trabajo:", TRABAJO)

### Interruptor para probar con un modelo real (opcional)

El notebook funciona entero con el modelo de juguete. Hay **una** celda opcional (Parte 6) que ejecuta el agente con un modelo real. Está apagada.

In [ ]:
USAR_MODELO_REAL = False             # ← True para probar la Parte 6 con el modelo de tu .env (LLM_PROVIDER)
EJECUTAR_LLAMADAS_DE_PAGO = False    # ← True para permitir proveedores de pago (anthropic/openai); LM Studio es gratis

---
# Parte 1 · El problema N×M: por qué existe MCP

## Antes de MCP: cada integración era artesanal

Imagina **N aplicaciones de IA** (un chat, un IDE, un agente interno...) y **M sistemas** a los que quieren llegar (un CRM, un repositorio, una base de datos, el correo). Sin un estándar, cada pareja necesita su propio código de conexión: **N × M integraciones**, cada una con su formato, su autenticación y sus errores.

| Sin estándar | Con un estándar |
| --- | --- |
| 3 apps × 4 sistemas = **12** conectores distintos | 3 clientes + 4 servidores = **7** piezas |
| Cada app reimplementa el CRM a su manera | El CRM se expone **una vez** como servidor |
| Cambia el CRM y se rompen N apps | Cambia el servidor; los clientes siguen igual |

Esa es la idea de **MCP (Model Context Protocol)**: un protocolo abierto que estandariza **cómo una aplicación de IA descubre y usa herramientas y datos externos**. La analogía clásica es **USB-C** (o, para quien viene de infraestructura, un **driver/CSI**: la app habla con una interfaz común y el "dispositivo" implementa los detalles). Otra analogía útil: **LSP** (Language Server Protocol) hizo lo mismo para los editores y los lenguajes.

## MCP no reemplaza el *tool calling*; lo **estandariza hacia afuera**

| | *Tool calling* tradicional (`@tool`) | MCP |
| --- | --- | --- |
| Dónde vive la herramienta | **Dentro** de tu aplicación (misma base de código) | En un **servidor** aparte (otro proceso, otro equipo, otra empresa) |
| Descubrimiento | Tú las registras en el código | El cliente las **descubre** (`tools/list`) al conectarse |
| Quién la mantiene | Tu equipo | Quien publica el servidor |
| Reutilización | Copiar código | Cualquier cliente MCP la usa |

## Ecosistema

| Dónde | Qué hay |
| --- | --- |
| Repositorio `modelcontextprotocol/servers` | Servidores de **referencia** (hora, archivos, `fetch`, git, memoria...) y ejemplos |
| **Registro oficial de MCP** | Catálogo de servidores publicados, con metadatos del paquete |
| Catálogos y *awesome lists* | Miles de servidores de la comunidad, de calidad muy variable: revisa quién los mantiene y cuándo se actualizaron |

> 💬 **Para pensar:** si tu agente ya tiene 5 herramientas `@tool`, ¿qué ganas y qué pierdes al exponerlas como un servidor MCP que otros clientes también puedan usar?

### Observación: lo que ve el modelo es siempre un esquema

Venga de `@tool` o de MCP, el modelo recibe lo mismo: **nombre + descripción + esquema JSON de los argumentos**. Veámoslo con una herramienta local.

In [ ]:
from langchain.tools import tool


@tool
def sumar(a: int, b: int) -> int:
    """Suma dos enteros."""
    return a + b


print(json.dumps(sumar.tool_call_schema.model_json_schema(), indent=2, ensure_ascii=False))

> ✍️ **Ejercicio 1.** Cuenta cuántos conectores necesitas para **5 clientes y 8 sistemas** con y sin estándar. ¿A partir de cuántos sistemas "vale la pena" un estándar si tienes 2 clientes?
>
> *Pista: sin estándar es N×M; con estándar, N+M.*

---
# Parte 2 · Arquitectura, transportes y ciclo de vida

## Tres roles

```
┌─────────────────────────── HOST ───────────────────────────┐
│  La aplicación de IA (IDE, chat, tu agente de LangChain)    │
│   ┌────────────┐   ┌────────────┐                           │
│   │ Cliente MCP│   │ Cliente MCP│   ← un cliente por servidor│
│   └─────┬──────┘   └─────┬──────┘                           │
└─────────┼────────────────┼──────────────────────────────────┘
          │ JSON-RPC       │ JSON-RPC
    ┌─────▼──────┐   ┌─────▼──────┐
    │ Servidor A │   │ Servidor B │   ← expone herramientas, datos, plantillas
    │ (HubSpot)  │   │ (hora)     │
    └────────────┘   └────────────┘
```

| Rol | Qué es | Ejemplo |
| --- | --- | --- |
| **Host** | La aplicación que el usuario usa y que **decide** a qué servidores conectarse | VS Code, Claude Desktop, Claude Code, tu agente |
| **Cliente** | Componente **dentro** del host que mantiene **una** conexión con **un** servidor | `ClientSession` del SDK, `MultiServerMCPClient` |
| **Servidor** | Programa que expone capacidades por MCP | El servidor de HubSpot que construirás hoy |

Todo viaja como **JSON-RPC 2.0** (mensajes JSON con `method`, `params` e `id`).

## Transportes: cómo viajan los mensajes

| Transporte | Cómo funciona | Cuándo |
| --- | --- | --- |
| **stdio** | El cliente **lanza** el servidor como subproceso y habla por `stdin`/`stdout` | Servidores **locales**, un solo usuario |
| **Streamable HTTP** | El servidor es un servicio web con un endpoint (`/mcp`); POST para enviar, opcionalmente SSE para transmitir | Servidores **remotos** y compartidos |
| ~~HTTP+SSE~~ | El transporte antiguo | **Deprecado**: migra a Streamable HTTP |

> ⚠️ **Regla de oro de stdio:** `stdout` es **el canal del protocolo**. Un solo `print()` de depuración lo corrompe. Los logs van **siempre a `stderr`** (lo verás en la Parte 9).

## Ciclo de vida (en `2025-11-25`)

1. **`initialize`**: el cliente propone una versión y declara sus capacidades; el servidor responde con la suya y las suyas.
2. **`notifications/initialized`**: el cliente confirma.
3. **Descubrimiento**: `tools/list`, `resources/list`, `prompts/list`.
4. **Uso**: `tools/call` (con los argumentos), `resources/read`, `prompts/get`.
5. **Cierre**: se cierra la conexión (con stdio, termina el subproceso).

> 📌 **Nota de versión.** La revisión `2026-07-28` de la especificación **elimina el `initialize` y las sesiones** del transporte HTTP (cada petición lleva su versión y capacidades) y **depreca** *sampling*, *roots* y *logging* por protocolo. El SDK que usamos (`mcp 1.30.0`) implementa todavía `2025-11-25`, con `initialize`; es lo que verás hoy. Los conceptos (tools, resources, prompts, transportes) **no cambian**.

Lo vamos a ver **en crudo**: escribimos un servidor diminuto y le hablamos en JSON-RPC a mano, sin SDK de cliente.

In [ ]:
SERVIDOR_DEMO = r"""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo")


@mcp.tool()
def sumar(a: int, b: int) -> int:
    "Suma dos enteros."
    return a + b


@mcp.resource("config://version")
def version() -> str:
    "Versión de la aplicación de ejemplo."
    return "1.0.0"


@mcp.prompt()
def resumir(texto: str) -> str:
    "Plantilla para pedir un resumen."
    return f"Resume en una frase: {texto}"


if __name__ == "__main__":
    mcp.run()   # transporte stdio por defecto
"""
(TRABAJO / "demo_server.py").write_text(SERVIDOR_DEMO, encoding="utf-8")
print("Escrito:", TRABAJO / "demo_server.py")

In [ ]:
import subprocess

proceso = subprocess.Popen(
    [sys.executable, str(TRABAJO / "demo_server.py")],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True,
)


def enviar(mensaje: dict, esperar_respuesta: bool = True):
    """Escribe UNA línea JSON en el stdin del servidor y (si es una petición) lee UNA línea de respuesta."""
    print("➡️ cliente:", json.dumps(mensaje, ensure_ascii=False)[:230])
    proceso.stdin.write(json.dumps(mensaje) + "\n")
    proceso.stdin.flush()
    if esperar_respuesta:
        respuesta = json.loads(proceso.stdout.readline())
        print("⬅️ servidor:", json.dumps(respuesta, ensure_ascii=False)[:330], "\n")
        return respuesta


# 1) initialize: versión y capacidades
enviar({"jsonrpc": "2.0", "id": 1, "method": "initialize",
        "params": {"protocolVersion": "2025-11-25", "capabilities": {}, "clientInfo": {"name": "cliente-a-mano", "version": "0"}}})
# 2) notificación: no lleva id y no tiene respuesta
enviar({"jsonrpc": "2.0", "method": "notifications/initialized"}, esperar_respuesta=False)
# 3) descubrimiento
enviar({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
enviar({"jsonrpc": "2.0", "id": 3, "method": "resources/list"})
enviar({"jsonrpc": "2.0", "id": 4, "method": "prompts/list"})
# 4) uso
enviar({"jsonrpc": "2.0", "id": 5, "method": "tools/call", "params": {"name": "sumar", "arguments": {"a": 2, "b": 3}}})
# 5) cierre
proceso.stdin.close()
proceso.wait(timeout=10)
print("Código de salida del servidor:", proceso.returncode)

**Qué observar.**

- La respuesta a `initialize` trae `protocolVersion`, `capabilities` (qué primitivas soporta) y `serverInfo`.
- `tools/list` devuelve, por cada herramienta, su **nombre, descripción y `inputSchema`** (un esquema JSON): **es exactamente lo que verá el modelo**.
- `tools/call` devuelve `content` (bloques de texto) y `isError`. Un error **de la herramienta** viaja *dentro* de un resultado normal con `isError: true`; un error **del protocolo** (método inexistente) viaja como `error` JSON-RPC (lo verás en la Parte 9).
- Las **notificaciones** no llevan `id` ni tienen respuesta.

---
# Parte 3 · Las tres primitivas: tools, resources y prompts

Un servidor puede ofrecer tres tipos de cosas. Lo que los distingue es **quién decide usarlas**:

| Primitiva | La controla | Para qué | Ejemplo | Cómo se ve en un cliente real |
| --- | --- | --- | --- | --- |
| **Tools** (herramientas) | El **modelo** | Acciones y consultas | `search_contacts`, `add_note` | El modelo las llama solo |
| **Resources** (recursos) | La **aplicación** (o la persona, al elegirlos) | Datos de solo lectura con una URI | `clientes://42`, `file:///reporte.md` | En Claude Code, `@servidor:uri` los adjunta al mensaje |
| **Prompts** (plantillas) | La **persona** | Flujos reutilizables con parámetros | "Revisa este código" | En Claude Code aparecen como comandos `/servidor:prompt` |

Los recursos pueden ser **fijos** (`docs://guia`) o **plantillas de URI** (`clientes://{id}`) cuyo parámetro se rellena al pedirlos. Del lado del **cliente**, el servidor puede además pedir tres cosas: **sampling** (usar el modelo del cliente), **consulta a la persona** (preguntarle un dato) y **roots** (conocer las carpetas con las que trabaja el cliente).

Escribimos un servidor con una de cada tipo y lo recorremos como lo haría un cliente.

In [ ]:
SERVIDOR_PRIMITIVAS = r"""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("primitivas")


# TOOL: la decide llamar el modelo. Los tipos y el docstring forman el esquema que lee.
@mcp.tool()
def convertir_a_quetzales(usd: float) -> str:
    "Convierte dólares a quetzales con una tasa fija de ejemplo (7.8)."
    return f"{usd} USD = {usd * 7.8:.2f} GTQ"


# RESOURCE fijo: un dato con URI propia.
@mcp.resource("docs://guia")
def guia() -> str:
    "Guía de estilo para las respuestas."
    return "Responde en español, con frases cortas y un ejemplo."


# RESOURCE con plantilla de URI: el {id} se rellena al pedirlo.
@mcp.resource("clientes://{id}")
def cliente(id: str) -> str:
    "Ficha de un cliente de ejemplo."
    return f"Cliente {id}: Cafetales del Norte, plan Pro."


# PROMPT: una plantilla con parámetros que la persona elige.
@mcp.prompt()
def revisar_codigo(lenguaje: str, codigo: str) -> str:
    "Plantilla para pedir una revisión de código."
    return f"Revisa este código en {lenguaje} y lista tres mejoras:\n\n{codigo}"
"""
(TRABAJO / "primitivas_server.py").write_text(SERVIDOR_PRIMITIVAS, encoding="utf-8")

import primitivas_server
from mcp.shared.memory import create_connected_server_and_client_session as conectar

print("Servidor escrito.")

In [ ]:
async with conectar(primitivas_server.mcp._mcp_server) as sesion:   # cliente y servidor en el mismo proceso
    print("── TOOLS ──")
    for t in (await sesion.list_tools()).tools:
        print(f"🔧 {t.name}: {t.description}  | entrada: {list(t.inputSchema['properties'])}")
    r = await sesion.call_tool("convertir_a_quetzales", {"usd": 25})
    print("   llamada →", r.content[0].text)

    print("\n── RESOURCES ──")
    for r in (await sesion.list_resources()).resources:
        print(f"📄 {r.uri}")
    for r in (await sesion.list_resource_templates()).resourceTemplates:
        print(f"📄 plantilla {r.uriTemplate}")
    print("   leer docs://guia →", (await sesion.read_resource("docs://guia")).contents[0].text)
    print("   leer clientes://42 →", (await sesion.read_resource("clientes://42")).contents[0].text)

    print("\n── PROMPTS ──")
    for p in (await sesion.list_prompts()).prompts:
        print(f"💬 {p.name}({', '.join(a.name for a in p.arguments)}): {p.description}")
    got = await sesion.get_prompt("revisar_codigo", {"lenguaje": "Python", "codigo": "print(1+1)"})
    print("   mensaje generado →", got.messages[0].role, "|", got.messages[0].content.text.replace("\n", " ⏎ "))

**Qué observar.**

- Cada primitiva tiene su **pareja de métodos**: `tools/list` + `tools/call`, `resources/list` + `resources/read`, `prompts/list` + `prompts/get`.
- Una **tool** *hace* algo y devuelve un resultado; un **resource** *es* un dato (no tiene efectos); un **prompt** *produce mensajes* listos para enviar al modelo, con tus parámetros.
- La plantilla `clientes://{id}` se anuncia con `resources/templates/list`, no con `resources/list`.

### Y desde LangChain

`langchain-mcp-adapters` convierte las **tools** en `@tool` (lo veremos en la Parte 6). Los recursos y los prompts llegan como **blobs** y **mensajes** que tú decides cómo usar:

In [ ]:
from langchain_mcp_adapters.prompts import load_mcp_prompt
from langchain_mcp_adapters.resources import load_mcp_resources

async with conectar(primitivas_server.mcp._mcp_server) as sesion:
    blobs = await load_mcp_resources(sesion, uris=["docs://guia", "clientes://42"])
    for b in blobs:
        print("Blob:", b.metadata["uri"], "→", b.as_string())
    mensajes = await load_mcp_prompt(sesion, "revisar_codigo", arguments={"lenguaje": "SQL", "codigo": "select * from t"})
    print("\nMensajes de LangChain:", [(type(m).__name__, m.text[:40] + "...") for m in mensajes])

> 💬 **Para pensar:** ¿cuál de las tres primitivas usarías para (a) darle al agente el manual de estilo de tu empresa como contexto, (b) un botón "preparar mi llamada de ventas", (c) crear un ticket? Justifica según **quién** decide usarla.
>
> ✍️ **Ejercicio 2.** Agrega al servidor un recurso `docs://glosario` y un prompt `traducir(texto, idioma)`; vuelve a ejecutar la celda anterior y comprueba que aparecen.

---
# Parte 4 · Explorar un servidor existente

Usaremos un servidor **oficial y simple**: **`mcp-server-time`** (hora y zonas horarias), del repositorio de referencia `modelcontextprotocol/servers`. No toca archivos: es de solo lectura, ideal para empezar.

Dos formas de explorar un servidor:

| Herramienta | Qué es | Comando |
| --- | --- | --- |
| **MCP Inspector** | Interfaz web oficial para listar y probar tools, resources y prompts y ver las notificaciones | `npx @modelcontextprotocol/inspector uvx mcp-server-time==2026.11.11` |
| **Cliente mínimo en Python** | Unas líneas con el SDK (lo que haremos aquí) | Celda siguiente |

> 📌 El Inspector es un paquete de Node.js. En noviembre de 2026 la versión `2.15.0` exige **Node 22.23 o superior**; si tienes Node 20, el comando falla: usa el cliente de Python. (Sin probar en este equipo por esa razón.)

> 📌 **Fijar la versión** (`==2026.11.11`) es una buena práctica: ejecutas **exactamente** lo que probaste. `uvx mcp-server-time` sin versión descargaría "lo último" cada vez.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVIDOR_HORA = StdioServerParameters(command="uvx", args=["mcp-server-time==2026.11.11"])


async def explorar(params):
    async with stdio_client(params) as (leer, escribir), ClientSession(leer, escribir) as sesion:
        init = await sesion.initialize()
        print(f"Servidor: {init.serverInfo.name} · protocolo {init.protocolVersion}")
        print("Capacidades:", sorted(init.capabilities.model_dump(exclude_none=True)))
        for t in (await sesion.list_tools()).tools:
            anot = t.annotations.model_dump(exclude_none=True) if t.annotations else {}
            print(f"\n🔧 {t.name}: {t.description.splitlines()[0]}")
            print("   anotaciones:", anot)


if shutil.which("uvx"):
    await explorar(SERVIDOR_HORA)
else:
    print("⏭️ Omitida: no está `uvx` (viene con uv). Instala uv para ejecutar este servidor de referencia.")

In [ ]:
async def llamar_hora():
    async with stdio_client(SERVIDOR_HORA) as (leer, escribir), ClientSession(leer, escribir) as sesion:
        await sesion.initialize()
        r = await sesion.call_tool("get_current_time", {"timezone": "America/Guatemala"})
        print(r.content[0].text)
        r = await sesion.call_tool("get_current_time", {"timezone": "Zona/Inventada"})
        print("\n¿Error?", r.isError, "→", r.content[0].text[:120])


if shutil.which("uvx"):
    await llamar_hora()

**Qué observar.**

- Solo **tools**: este servidor no ofrece resources ni prompts (`capabilities` lo dice).
- Las **anotaciones** (`readOnlyHint`, `destructiveHint`, `idempotentHint`, `openWorldHint`) son **metadatos** que describen la herramienta al cliente: "solo lee", "repetirla da lo mismo", "toca el mundo exterior". Un cliente puede usarlas para decidir cómo mostrarla.
- Ante un argumento inválido, el servidor responde con `isError=True` y un mensaje: el modelo puede **leerlo y corregirse**.

> ✍️ **Ejercicio 3.** Ejecuta el Inspector si tienes Node 22+. Lista las herramientas y llama `convert_time` desde la interfaz. ¿Qué ves en la pestaña de notificaciones?

---
# Parte 5 · Construir tu propio servidor MCP (un CRM)

Con `FastMCP` un servidor es **funciones decoradas**: el nombre, el docstring y los **tipos** se convierten en el esquema que ve el modelo. Nuestro caso es un **CRM** (HubSpot en la práctica; aquí, una versión simulada en memoria con **datos ficticios**) con tres herramientas, dos de **lectura** y una de **escritura**, más un recurso y un prompt:

| Pieza | Tipo | Qué hace |
| --- | --- | --- |
| `search_contacts(query, limit)` | Tool · lectura | Busca contactos por texto |
| `get_contact_notes(contact_id, limit)` | Tool · lectura | Lista las notas de un contacto |
| `add_note(contact_id, body)` | Tool · **escritura** | Crea una nota asociada a un contacto |
| `crm://contacto/{contact_id}` | Resource | Ficha del contacto en Markdown |
| `preparar_llamada(contact_id)` | Prompt | "Prepara mi llamada con este contacto" |

Tres detalles de diseño:

- **Los tipos con restricciones son el esquema.** `ContactId` con un patrón de dígitos y `Limit` entre 1 y 10 aparecen en el JSON Schema que lee el modelo **y** los valida el SDK antes de llamar a tu función. Un argumento inválido vuelve como error legible.
- **Anotaciones como metadato.** `readOnlyHint=True` en las lecturas; `add_note` declara que **no** es solo lectura ni idempotente (cada llamada crea una nota).
- **Logs a `stderr`**, nunca `print()` (regla de stdio).

In [ ]:
CRM_SERVER = r"""
import logging
import sys
from typing import Annotated, Any

from mcp.server.fastmcp import FastMCP
from mcp.types import ToolAnnotations
from pydantic import Field

logging.basicConfig(stream=sys.stderr, level=logging.INFO, format="%(levelname)s %(message)s")   # stderr, no stdout
log = logging.getLogger("crm")

mcp = FastMCP("crm", instructions="CRM de ejemplo: busca contactos, lee sus notas y puede agregar una nota.")

CONTACTOS = {}
NOTAS = {}
ESCRITURAS = []


def reiniciar():
    CONTACTOS.clear(); NOTAS.clear(); ESCRITURAS.clear()
    for id_, nombre, apellido, correo, empresa in [
        ("101", "Ana", "Morales", "ana.morales@example.com", "Cafetales del Norte"),
        ("102", "Luis", "Pérez", "luis.perez@example.org", "Transportes Quetzal"),
        ("103", "Marta", "Solís", "marta.solis@example.com", "Cafetales del Norte"),
        ("104", "Diego", "Ruiz", "diego.ruiz@example.net", "Textiles Atitlán"),
    ]:
        CONTACTOS[id_] = {"id": id_, "name": f"{nombre} {apellido}", "email": correo, "company": empresa}
    NOTAS.update({"101": ["Pidió cotización de 200 bolsas. Prefiere contacto por la mañana."],
                  "102": ["Llamada de seguimiento: confirma que renovará el contrato en enero."],
                  "103": ["Envió los datos de facturación. Pendiente: confirmar NIT."], "104": []})


reiniciar()

# Los tipos con restricciones SON el esquema que ve el modelo, y el SDK los valida antes de llamar a la función.
ContactId = Annotated[str, Field(pattern=r"^\d{1,15}$", description="Id numérico del contacto (lo devuelve search_contacts).")]
Limit = Annotated[int, Field(ge=1, le=10, description="Máximo de resultados (1 a 10).")]

# Anotaciones: METADATO que describe la herramienta al cliente.
LECTURA = ToolAnnotations(readOnlyHint=True, destructiveHint=False, idempotentHint=True, openWorldHint=True)
ESCRITURA = ToolAnnotations(readOnlyHint=False, destructiveHint=False, idempotentHint=False, openWorldHint=True)


@mcp.tool(annotations=LECTURA)
def search_contacts(query: Annotated[str, Field(min_length=2, max_length=80)], limit: Limit = 5) -> dict[str, Any]:
    "Busca contactos por nombre, empresa o correo."
    q = query.lower()
    filas = [c for c in CONTACTOS.values() if q in " ".join(c.values()).lower()][:limit]
    log.info("search_contacts(%r) -> %d resultados", query, len(filas))
    return {"contacts": filas, "count": len(filas)}


@mcp.tool(annotations=LECTURA)
def get_contact_notes(contact_id: ContactId, limit: Limit = 5) -> dict[str, Any]:
    "Notas de un contacto."
    return {"notes": NOTAS.get(contact_id, [])[:limit]}


@mcp.tool(annotations=ESCRITURA)
def add_note(contact_id: ContactId, body: Annotated[str, Field(min_length=3, max_length=2000)]) -> dict[str, Any]:
    "Crea una nota asociada a un contacto. ESCRIBE en el CRM (cada llamada crea una nota nueva)."
    if contact_id not in CONTACTOS:
        raise ValueError(f"El contacto {contact_id} no existe")
    NOTAS.setdefault(contact_id, []).append(body)
    ESCRITURAS.append({"contact_id": contact_id, "body": body})
    log.info("add_note(%s) -> nota creada", contact_id)
    return {"created": True, "contact_id": contact_id}


@mcp.resource("crm://contacto/{contact_id}", mime_type="text/markdown")
def ficha(contact_id: str) -> str:
    "Ficha de un contacto en Markdown."
    c = CONTACTOS[contact_id]
    return f"# {c['name']}\n\n- Empresa: {c['company']}\n- Correo: {c['email']}\n"


@mcp.prompt()
def preparar_llamada(contact_id: str) -> str:
    "Prepara una llamada de ventas con un contacto."
    return f"Prepara mi llamada con el contacto {contact_id}: busca sus notas y dame tres viñetas y dos preguntas."


if __name__ == "__main__":
    mcp.run()
"""
(TRABAJO / "crm_server.py").write_text(CRM_SERVER, encoding="utf-8")

import crm_server   # también como módulo, para inspeccionar el "CRM" desde el notebook

print("Servidor escrito.", len(CRM_SERVER.splitlines()), "líneas.")

In [ ]:
async def probar_por_stdio(herramienta: str, argumentos: dict):
    """Lanza el servidor como subproceso (stdio), llama UNA herramienta y lo cierra."""
    params = StdioServerParameters(command=sys.executable, args=[str(TRABAJO / "crm_server.py")])
    async with stdio_client(params) as (leer, escribir), ClientSession(leer, escribir) as sesion:
        await sesion.initialize()
        return await sesion.call_tool(herramienta, argumentos)


r = await probar_por_stdio("search_contacts", {"query": "Cafetales"})
print("isError:", r.isError)
print(r.content[0].text[:300])

In [ ]:
async with conectar(crm_server.mcp._mcp_server) as sesion:
    herr = {t.name: t for t in (await sesion.list_tools()).tools}
    print("Esquema de add_note:\n", json.dumps(herr["add_note"].inputSchema["properties"], indent=1, ensure_ascii=False))
    print("\nAnotaciones:", {n: (t.annotations.readOnlyHint, t.annotations.idempotentHint) for n, t in herr.items()}, "(readOnly, idempotente)")

    print("\n¿Qué pasa con argumentos inválidos? El error vuelve al modelo como TEXTO legible:")
    for args in [{"contact_id": "abc", "body": "Hola mundo"}, {"contact_id": "101", "body": "x"}, {"contact_id": "999", "body": "Hola mundo"}]:
        r = await sesion.call_tool("add_note", args)
        print(f"   {args} → isError={r.isError}: {r.content[0].text.splitlines()[0][:100]}")
    print("   (escrituras reales hasta ahora:", len(crm_server.ESCRITURAS), ")")

**Qué observar.**

- Con **tres funciones** ya tienes un servidor MCP con descubrimiento, esquemas, transporte, un recurso y un prompt.
- Los **tipos** se convierten en JSON Schema (`pattern`, `minLength`...) y se **validan antes** de ejecutar tu función: los tres intentos inválidos no escribieron nada.
- Los errores no rompen la conexión: llegan como `isError=True` con un mensaje que el modelo puede leer y corregir.

> ✍️ **Ejercicio 4.** Cambia `Limit` para permitir hasta 20 resultados y vuelve a escribir el servidor. ¿Cambia el esquema que ve el modelo? Compruébalo con `tools/list`.

---
# Parte 6 · El agente con `langchain-mcp-adapters`

Ya tenemos un servidor. Ahora lo consume un **agente de LangChain** (el `create_agent` que viste en la S8 y la S9). La pieza nueva es `langchain-mcp-adapters`: convierte cada **tool** MCP en una `@tool` de LangChain. Para el agente no hay diferencia entre una herramienta local y una que vive en un servidor MCP.

```python
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({"crm": {"transport": "stdio", "command": "python", "args": ["crm_server.py"]}})
tools = await client.get_tools()             # tools MCP → @tool de LangChain
agent = create_agent(model, tools=tools)
```

En esta parte usamos el modo **en memoria** (cliente y servidor en el mismo proceso) para poder inspeccionar el CRM después; el resultado para el agente es idéntico. Y un **modelo de juguete** que decide con reglas fijas:

In [ ]:
import re
from contextlib import asynccontextmanager

from langchain.agents import create_agent
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_mcp_adapters.tools import load_mcp_tools


def llamada(nombre, args, n):
    return AIMessage(content="", tool_calls=[{"name": nombre, "args": args, "id": f"c{n}"}])


class ModeloGuionado(BaseChatModel):
    """Modelo de juguete: sin red ni costo. Entiende 3 peticiones (resumir notas, agregar nota, preguntar la hora)."""

    tool_names: list[str] = []

    @property
    def _llm_type(self) -> str:
        return "guionado"

    def bind_tools(self, tools, **kwargs):          # recibe las herramientas MCP ya convertidas en @tool
        return self.model_copy(update={"tool_names": [t.name for t in tools]})

    def _herramienta(self, corto):                  # con varios servidores los nombres llevan prefijo: "crm_search_contacts"
        return next((n for n in self.tool_names if n == corto or n.endswith("_" + corto)), corto)

    def _generate(self, mensajes, stop=None, run_manager=None, **kwargs):
        inicio = max(i for i, m in enumerate(mensajes) if isinstance(m, HumanMessage))    # el turno actual empieza en el último mensaje humano
        pregunta, ultimo, n = mensajes[inicio].text, mensajes[-1], len(mensajes)
        if isinstance(ultimo, HumanMessage):
            if "hora" in pregunta.lower():
                zona = re.search(r"en ([A-Z]\w+/\w+)", pregunta)
                r = llamada(self._herramienta("get_current_time"), {"timezone": zona.group(1) if zona else "America/Guatemala"}, n)
            elif "hablábamos" in pregunta:                       # MEMORIA: lo que hay en el historial de la sesión
                antes = [m for m in mensajes[:inicio] if isinstance(m, ToolMessage) and m.name.endswith("search_contacts")]
                nombre = re.search(r'"name":\s*"([^"]+)"', antes[-1].text).group(1) if antes else None
                r = AIMessage(content=f"Hablábamos de {nombre}." if nombre else "Todavía no hemos hablado de nadie en esta sesión.")
            else:
                quien = re.search(r"(?:notas de|nota a)\s+([A-ZÁÉÍÓÚ]\w+)", pregunta)
                r = llamada(self._herramienta("search_contacts"), {"query": quien.group(1) if quien else "Ana"}, n)
        elif ultimo.name.endswith("search_contacts"):
            cid = re.search(r'"id":\s*"(\d+)"', ultimo.text).group(1)
            if "nota a" in pregunta.lower():
                texto = re.search(r":\s*['\"“](.+?)['\"”]", pregunta).group(1)
                r = llamada(self._herramienta("add_note"), {"contact_id": cid, "body": texto}, n)
            else:
                r = llamada(self._herramienta("get_contact_notes"), {"contact_id": cid}, n)
        elif ultimo.name.endswith("get_contact_notes"):
            notas = re.findall(r'"([A-ZÁÉÍÓÚ][^"]{15,})"', ultimo.text)       # las notas de ese contacto son los textos largos
            r = AIMessage(content="Resumen de las notas: " + " | ".join(notas) if notas else "Ese contacto no tiene notas.")
        elif ultimo.name.endswith("add_note"):
            r = AIMessage(content="Listo, la nota quedó guardada." if "created" in ultimo.text else "No se guardó: " + ultimo.text[:100])
        else:
            r = AIMessage(content="Resultado: " + ultimo.text.replace("\n", " ")[:160])
        return ChatResult(generations=[ChatGeneration(message=r)])


@asynccontextmanager
async def herramientas(modulo):
    """Conecta un cliente MCP a un servidor del mismo proceso y entrega sus tools como `@tool` de LangChain."""
    async with conectar(modulo.mcp._mcp_server) as sesion:
        yield await load_mcp_tools(sesion)


def mostrar_traza(mensajes):
    for m in mensajes:
        if m.type == "human":
            print("👤", m.text)
        elif m.type == "ai":
            for c in m.tool_calls:
                print("🤖 pide", c["name"], c["args"])
        elif m.type == "tool":
            print("🔧", m.text.replace("\n", " ")[:170])
        if m.type == "ai" and not m.tool_calls:
            print("💬", m.text)

In [ ]:
crm_server.reiniciar()

async with herramientas(crm_server) as tools:
    print("Herramientas que ve el agente:", [(t.name, type(t).__name__) for t in tools])
    agente = create_agent(ModeloGuionado(), tools=tools, system_prompt="Eres un asistente de ventas. Responde en español.")
    resultado = await agente.ainvoke({"messages": [{"role": "user", "content": "Resume las notas de Luis."}]})

mostrar_traza(resultado["messages"])

**Qué observar.** La traza (👤 🤖 🔧) es la misma que ya conoces. Las herramientas son `StructuredTool` de LangChain; que por debajo hablen MCP con un servidor es un detalle que el agente no ve. Lo que cambió: **tú no escribiste esas herramientas**, las descubrió el cliente con `tools/list`.

Dos cosas que el adaptador te da:

- Las **anotaciones** de MCP llegan como `tool.metadata` (por ejemplo `readOnlyHint`).
- Si el servidor responde `isError`, el agente recibe el mensaje como un `ToolMessage` y puede corregirse.

In [ ]:
async with herramientas(crm_server) as tools:
    for t in tools:
        print(f"{t.name:<18} metadata: {t.metadata}")

## Escribir con aprobación humana (repaso de la S9)

`add_note` **modifica** el CRM. Para herramientas que modifican, la práctica básica que ya viste es el `HumanInTheLoopMiddleware`: el agente **se pausa** antes de ejecutar, una persona **aprueba o rechaza**, y el agente continúa. Necesita un *checkpointer* y un `thread_id`. Funciona igual con herramientas MCP, porque para LangChain son `@tool`.

In [ ]:
import uuid

from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


async def pedir_nota(decision):
    crm_server.reiniciar()
    async with herramientas(crm_server) as tools:
        agente = create_agent(
            ModeloGuionado(), tools=tools, checkpointer=InMemorySaver(),
            middleware=[HumanInTheLoopMiddleware(interrupt_on={"add_note": {"allowed_decisions": ["approve", "reject"]}})],
        )
        config = {"configurable": {"thread_id": str(uuid.uuid4())}}
        r = await agente.ainvoke({"messages": [{"role": "user", "content": "Agrega una nota a Ana: 'Enviar la propuesta el lunes'"}]}, config)
        while r.get("__interrupt__"):
            pedido = r["__interrupt__"][0].value["action_requests"][0]
            print(f"   ⏸️ el agente pide: {pedido['name']}({pedido['args']})  →  persona: {decision}")
            orden = {"type": "approve"} if decision == "aprobar" else {"type": "reject", "message": "No autorizado."}
            r = await agente.ainvoke(Command(resume={"decisions": [orden]}), config)
    print("   💬", r["messages"][-1].text, "| escrituras en el CRM:", len(crm_server.ESCRITURAS))


print("La persona aprueba:")
await pedir_nota("aprobar")
print("\nLa persona rechaza:")
await pedir_nota("rechazar")

**Qué observar.** Con "aprobar" la nota se escribe (1 escritura); con "rechazar", el agente recibe un mensaje de rechazo y **no se ejecuta nada** (0 escrituras). Las lecturas (`search_contacts`, `get_contact_notes`) no piden aprobación porque no están en `interrupt_on`.

In [ ]:
# 💰 FYI: esta celda consume dinero con anthropic/openai (≈ US$ 0.002). Con LM Studio (local) es gratis.
import os

from langchain.chat_models import init_chat_model

if USAR_MODELO_REAL:
    proveedor = os.getenv("LLM_PROVIDER", "openai").lower()
    if proveedor != "local" and not EJECUTAR_LLAMADAS_DE_PAGO:
        print("⏭️ Omitida: activa EJECUTAR_LLAMADAS_DE_PAGO para usar un proveedor de pago.")
    else:
        if proveedor == "local":
            real = init_chat_model(os.getenv("LOCAL_LLM_MODEL", "google/gemma-4-12b-qat"), model_provider="openai",
                                   base_url=os.getenv("LOCAL_LLM_URL", "http://localhost:1235/v1"), api_key="lm-studio", temperature=0)
        else:
            real = init_chat_model({"anthropic": "claude-haiku-5-5", "openai": "gpt-6-luna"}[proveedor], model_provider=proveedor)
        crm_server.reiniciar()
        async with herramientas(crm_server) as tools:
            agente_real = create_agent(real, tools=tools, system_prompt="Eres un asistente de ventas. Responde en español.")
            r = await agente_real.ainvoke({"messages": [{"role": "user", "content": "¿Qué dicen las notas de Luis?"}]})
        mostrar_traza(r["messages"])
else:
    print("(Modelo real desactivado: pon USAR_MODELO_REAL = True para probarlo.)")

---
# Parte 7 · Dos servidores a la vez (multi-servidor)

Un **host** puede tener **varios clientes**, uno por servidor. `MultiServerMCPClient` lo hace por ti: le das un diccionario de servidores y devuelve **todas** las herramientas juntas, para que **un mismo agente** use las de ambos. Aquí combinamos **nuestro CRM** con el servidor de referencia de **hora** de la Parte 4 (ya hecho por otros: no escribimos una línea).

Detalle práctico: si dos servidores tienen una herramienta con el mismo nombre, chocarían. `tool_name_prefix=True` antepone el nombre del servidor (`crm_search_contacts`, `hora_get_current_time`).

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

RELOJ_LOCAL = r"""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("reloj")


@mcp.tool()
def get_current_time(timezone: str) -> str:
    "Hora actual de una zona (de mentira, solo para cuando no hay uvx)."
    return f"2026-11-19T10:00:00 en {timezone}"


if __name__ == "__main__":
    mcp.run()
"""
(TRABAJO / "reloj_server.py").write_text(RELOJ_LOCAL, encoding="utf-8")

# El segundo servidor: el oficial de hora (vía uvx) o, si no hay uvx, el reloj local de arriba.
if shutil.which("uvx"):
    segundo = {"transport": "stdio", "command": "uvx", "args": ["mcp-server-time==2026.11.11"]}
else:
    segundo = {"transport": "stdio", "command": sys.executable, "args": [str(TRABAJO / "reloj_server.py")]}

cliente = MultiServerMCPClient(
    {
        "crm": {"transport": "stdio", "command": sys.executable, "args": [str(TRABAJO / "crm_server.py")]},
        "hora": segundo,
    },
    tool_name_prefix=True,
)
tools = await cliente.get_tools()
print("Herramientas de los dos servidores:")
for t in tools:
    print("  ", t.name)

In [ ]:
agente = create_agent(ModeloGuionado(), tools=tools, system_prompt="Eres un asistente de ventas. Responde en español.")

for pregunta in ["Resume las notas de Ana.", "¿Qué hora es en America/Guatemala?"]:
    r = await agente.ainvoke({"messages": [{"role": "user", "content": pregunta}]})
    mostrar_traza(r["messages"])
    print()

**Qué observar.** Un solo agente, **dos procesos servidor** (cada uno lanzado por stdio) y la herramienta correcta para cada pregunta: el CRM para las notas, el reloj para la hora. Cada herramienta lleva el prefijo de su servidor.

> 💬 **Para pensar:** con 5 servidores y 40 herramientas, ¿qué le pasa al contexto del modelo? Cada descripción y cada esquema ocupan tokens en **cada** llamada. Conecta solo lo que el agente necesita.

---
# Parte 8 · Registrar tu servidor en un cliente MCP real

Tu servidor no es solo para tu agente: **cualquier cliente MCP** puede usarlo. Se registra en un archivo de configuración que dice **cómo lanzarlo** (stdio) o **dónde está** (HTTP).

| Cliente | Dónde se configura | Cómo |
| --- | --- | --- |
| **Claude Code** (CLI) | `.mcp.json` en la raíz del proyecto (alcance *project*), o `~/.claude.json` (alcance *local*/*user*) | `claude mcp add ...` escribe el archivo; pide aprobación la primera vez que se usa uno de proyecto |
| **Claude Desktop** | `claude_desktop_config.json`: macOS `~/Library/Application Support/Claude/`, Windows `%APPDATA%\Claude\`, Linux `~/.config/Claude/` | *Settings → Developer → Edit Config*; hay que **reiniciar** la aplicación |

Formato (idéntico en lo esencial): una clave `mcpServers` con un objeto por servidor. **stdio** lleva `command`, `args` y `env`; **HTTP** lleva `type: "http"` y `url`. Se generan aquí con **rutas absolutas** (el directorio de trabajo del cliente puede no ser el tuyo).

In [ ]:
configuracion = {
    "mcpServers": {
        "crm": {                                     # stdio: el cliente LANZA el servidor
            "command": sys.executable,
            "args": [str(TRABAJO / "crm_server.py")],
            "env": {},                               # un servidor stdio recibe un entorno básico: aquí van sus variables
        },
        "crm-http": {                                # HTTP: el servidor ya está corriendo en alguna parte
            "type": "http",
            "url": "http://127.0.0.1:8000/mcp",
        },
    }
}
(TRABAJO / ".mcp.json").write_text(json.dumps(configuracion, indent=2), encoding="utf-8")
print(json.dumps(configuracion, indent=2))

print("\nEl mismo registro con la línea de comandos de Claude Code:")
print(f"  claude mcp add --transport stdio --scope project crm -- {sys.executable} {TRABAJO / 'crm_server.py'}")
print( "  claude mcp add --transport http  --scope project crm-http http://127.0.0.1:8000/mcp")
print( "  claude mcp list")

**Qué usar ya dentro del cliente** (Claude Code, según su documentación):

| Primitiva | Cómo se usa |
| --- | --- |
| Tools | El modelo las llama solo; se nombran `mcp__<servidor>__<herramienta>` (por ejemplo `mcp__crm__search_contacts`) |
| Resources | Escribe `@crm:crm://contacto/101` en tu mensaje y se adjunta la ficha |
| Prompts | Aparecen como comandos `/crm:preparar_llamada` |

Y `/mcp` dentro de la sesión muestra el estado de cada servidor.

> 📌 Si el servidor no aparece: revisa que el JSON sea válido, que las rutas sean **absolutas**, que `command` exista (prueba ejecutarlo a mano) y reinicia el cliente. Los logs del servidor (su `stderr`) los guarda el cliente: es lo primero que se mira (Parte 9).

> ✍️ **Ejercicio 5.** Copia el archivo generado a la raíz de una carpeta de prueba, abre Claude Code ahí, aprueba el servidor `crm` y pídele "busca a Ana en el CRM". ¿Qué nombre de herramienta ves en la traza?

---
# Parte 9 · Depuración

Cuando un servidor MCP "no funciona", casi siempre es una de cuatro cosas: **no arranca**, **no habla bien el protocolo**, **una herramienta falla** o **el cliente no lo encuentra**. Tus herramientas, de la más cercana al código a la más cercana al cliente:

| Herramienta | Para qué |
| --- | --- |
| **Logs a `stderr`** | Ver qué hace el servidor mientras corre |
| **Cliente mínimo** (`mini_client.py`, la Parte 4) | Comprobar que el servidor arranca, lista sus herramientas y responde |
| **MCP Inspector** | Interfaz web para listar y probar tools, resources y prompts, y ver las notificaciones |
| **Logs del cliente** | Qué vio la aplicación (en Claude Desktop: `mcp.log` y `mcp-server-<nombre>.log` en `~/Library/Logs/Claude`, `%APPDATA%\Claude\logs` o `~/.config/Claude/logs`) |
| **`curl` y los logs del contenedor** | Para servidores HTTP (su `stderr` no llega al cliente) |

## Primer hábito: ver el protocolo en crudo

Una función que lanza un servidor, le manda mensajes y devuelve **lo que escribió en `stdout`** (el protocolo) y **en `stderr`** (los logs):

In [ ]:
INIT = {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-11-25", "capabilities": {}, "clientInfo": {"name": "depurador", "version": "0"}}}
INITIALIZED = {"jsonrpc": "2.0", "method": "notifications/initialized"}


def hablar(archivo: str, *mensajes: dict):
    """Lanza el servidor, le envía los mensajes y devuelve (líneas de stdout, texto de stderr)."""
    entrada = "\n".join(json.dumps(m) for m in (INIT, INITIALIZED, *mensajes)) + "\n"
    p = subprocess.run([sys.executable, str(TRABAJO / archivo)], input=entrada, capture_output=True, text=True, timeout=30)
    return p.stdout.splitlines(), p.stderr


salida, logs = hablar("crm_server.py", {"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": "search_contacts", "arguments": {"query": "Ana"}}})
print("stdout (PROTOCOLO): líneas =", len(salida), "→ todas son JSON:", all(json.loads(l) for l in salida))
print("stderr (LOGS):")
print(logs)

**Qué observar.** `stdout` solo trae mensajes JSON-RPC; los **logs** (`INFO search_contacts('Ana') -> 1 resultados`) salieron por `stderr`. El cliente que lanza el servidor captura `stderr` y lo guarda o lo muestra; **no** lo mezcla con el protocolo.

## Error clásico: un `print()` en `stdout`

Un servidor con un `print("arrancando...")` antes de `mcp.run()`. Lo que el cliente espera leer en `stdout` es una línea JSON y se encuentra con texto:

In [ ]:
RUIDOSO = r"""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("ruidoso")

@mcp.tool()
def ping() -> str:
    "Responde pong."
    return "pong"

print("arrancando el servidor...", flush=True)     # ← ERROR: stdout es el protocolo
mcp.run()
"""
(TRABAJO / "ruidoso_server.py").write_text(RUIDOSO, encoding="utf-8")

salida, logs = hablar("ruidoso_server.py")
for i, linea in enumerate(salida):
    try:
        json.loads(linea)
        print(f"línea {i}: ✅ JSON válido  {linea[:70]}")
    except json.JSONDecodeError:
        print(f"línea {i}: ⛔ NO es JSON → {linea!r}   (el cliente falla o se desincroniza al leerla)")

**La solución:** `print("...", file=sys.stderr)` o, mejor, el módulo `logging` configurado con `stream=sys.stderr` (como en `crm_server.py`). Es una regla **del protocolo** de stdio, no una recomendación de estilo.

## Dos tipos de error

| | Error **de la herramienta** | Error **del protocolo** |
| --- | --- | --- |
| Ejemplo | `contact_id` inexistente, argumento inválido | Un método que no existe |
| Cómo viaja | Resultado normal con `isError: true` y un mensaje | Mensaje `error` de JSON-RPC (con `code`) |
| Quién lo ve | **El modelo** (puede corregirse) | El cliente (el desarrollador) |

In [ ]:
salida, _ = hablar(
    "crm_server.py",
    {"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": "add_note", "arguments": {"contact_id": "999", "body": "Hola mundo"}}},
    {"jsonrpc": "2.0", "id": 3, "method": "metodo/inexistente"},
)
for linea in salida[1:]:
    m = json.loads(linea)
    if "error" in m:
        print(f"id {m['id']} → ERROR DE PROTOCOLO  code={m['error']['code']}: {m['error']['message']}")
    else:
        print(f"id {m['id']} → resultado normal  isError={m['result']['isError']}: {m['result']['content'][0]['text'][:70]}")

## Inspector, HTTP y el cliente

- **Inspector:** `npx @modelcontextprotocol/inspector <comando del servidor>` (por ejemplo `npx @modelcontextprotocol/inspector uv run python mcp_server.py`). Abre una interfaz web donde **listas y llamas** tools, resources y prompts y ves las notificaciones. Requiere Node 22.23+ (versión `2.15.0`).
- **Servidor HTTP:** `curl` al endpoint y los **logs del contenedor** (`docker compose logs mcp`): su `stderr` no viaja al cliente. El cliente mínimo también habla HTTP: `uv run python mini_client.py --url http://127.0.0.1:8000/mcp`.
- **Cuando el cliente no ve el servidor:** ejecuta el `command` **a mano** en una terminal; usa rutas **absolutas**; recuerda que el servidor stdio recibe solo un entorno básico, así que pon en `env` lo que necesite; reinicia el cliente tras cambiar el archivo de configuración.

> ✍️ **Ejercicio 6.** Agrega un `print("depurando")` a `crm_server.py` (en `stdout`), vuelve a ejecutar `hablar(...)` y mira qué línea rompe el protocolo. Luego cámbialo por `log.info(...)`.

---
# Parte 10 · Servir el agente: API, sesiones e interfaz

Hasta ahora el agente corre dentro de un notebook. Para que lo use **otra aplicación** (una interfaz web, otro servicio, una persona con un cliente HTTP) hay que **exponerlo con una API**.

```
 Bruno / curl ─┐
               ├─ HTTP ─►  API (FastAPI)  ──►  agente (create_agent + tools MCP)  ──►  servidor MCP  ──►  CRM
 Streamlit ────┘                │
                                └──►  SQLite (checkpointer): el HISTORIAL de cada sesión
```

## La idea central

> **El agente es un servicio sin estado en el proceso. El estado vive en el checkpointer, y el cliente solo lleva el id de sesión.**

Es como un servicio web con sesiones en una base de datos: el cliente conserva el identificador (la "cookie") y cualquier réplica del servicio puede continuar la conversación. Aquí:

| Pieza | Papel |
| --- | --- |
| `POST /chat` | Recibe `message` y, opcionalmente, `session_id`. **Sin id: crea una sesión y la devuelve.** Con id: continúa esa conversación |
| `session_id` | **Es el `thread_id`** del checkpointer. Mismo id = misma conversación |
| Checkpointer en **SQLite** | Guarda el historial en disco: la sesión **sobrevive a un reinicio** de la API (con `InMemorySaver` se perdería) |
| `status` | `completed` o `awaiting_approval` (el agente se pausó antes de una escritura) |
| `POST /sessions/{id}/approve` | `{"approved": true\|false}`: reanuda al agente con la decisión de la persona |
| `GET` / `DELETE /sessions/{id}` | Historial legible / borrar el hilo |

Montamos una versión mínima **en el notebook** y la probamos **sin levantar ningún servidor**, con `TestClient` (llama a la aplicación en el mismo proceso). La versión completa (modelos de entrada y salida, listado de sesiones, `usage`, errores `404`/`409`/`422`/`503`) está en `api.py` de la práctica.

In [ ]:
from contextlib import asynccontextmanager

logging.getLogger("httpx").setLevel(logging.WARNING)   # TestClient usa httpx: no mostrar cada petición

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from pydantic import BaseModel


class ChatIn(BaseModel):
    message: str
    session_id: str | None = None          # vacío = sesión nueva


class ChatOut(BaseModel):
    session_id: str
    status: str                            # "completed" | "awaiting_approval"
    answer: str
    pending_action: dict | None = None     # la herramienta y los argumentos que esperan aprobación


class ApproveIn(BaseModel):
    approved: bool


def crear_app(ruta_db: str) -> FastAPI:
    @asynccontextmanager
    async def lifespan(app):
        # Se abren UNA vez al arrancar: el checkpointer (SQLite) y la sesión con el servidor MCP.
        async with AsyncSqliteSaver.from_conn_string(ruta_db) as saver, conectar(crm_server.mcp._mcp_server) as sesion:
            tools = await load_mcp_tools(sesion)
            app.state.agente = create_agent(
                ModeloGuionado(), tools=tools, checkpointer=saver,
                middleware=[HumanInTheLoopMiddleware(interrupt_on={"add_note": {"allowed_decisions": ["approve", "reject"]}})],
            )
            app.state.saver = saver
            yield

    app = FastAPI(title="Agente MCP (demo)", lifespan=lifespan)

    def respuesta(session_id, resultado) -> ChatOut:
        if resultado.get("__interrupt__"):                       # el agente se pausó: hay una escritura por aprobar
            pedido = resultado["__interrupt__"][0].value["action_requests"][0]
            return ChatOut(session_id=session_id, status="awaiting_approval", answer="", pending_action={"tool": pedido["name"], "args": pedido["args"]})
        return ChatOut(session_id=session_id, status="completed", answer=resultado["messages"][-1].text)

    @app.post("/chat", response_model=ChatOut)
    async def chat(body: ChatIn):
        session_id = body.session_id or str(uuid.uuid4())         # el session_id ES el thread_id
        config = {"configurable": {"thread_id": session_id}}
        return respuesta(session_id, await app.state.agente.ainvoke({"messages": [{"role": "user", "content": body.message}]}, config))

    @app.post("/sessions/{session_id}/approve", response_model=ChatOut)
    async def approve(session_id: str, body: ApproveIn):
        config = {"configurable": {"thread_id": session_id}}
        decision = {"type": "approve"} if body.approved else {"type": "reject", "message": "No autorizado."}
        return respuesta(session_id, await app.state.agente.ainvoke(Command(resume={"decisions": [decision]}), config))

    @app.get("/sessions/{session_id}")
    async def historial(session_id: str):
        estado = await app.state.agente.aget_state({"configurable": {"thread_id": session_id}})
        if not estado.values:
            raise HTTPException(404, "Sesión inexistente")
        return [{"role": m.type, "content": m.text} for m in estado.values["messages"]]

    @app.delete("/sessions/{session_id}", status_code=204)
    async def borrar(session_id: str):
        await app.state.saver.adelete_thread(session_id)

    return app


print("Rutas:", sorted(r.path for r in crear_app(":memory:").routes if r.path.startswith(("/chat", "/sessions"))))

Probamos el recorrido completo con `TestClient`. Cada `with TestClient(app)` equivale a **arrancar** la API (se ejecuta el `lifespan`) y al salir, a **detenerla**.

In [ ]:
db = TRABAJO / "sesiones.db"      # el archivo SQLite que guarda las sesiones
crm_server.reiniciar()

with TestClient(crear_app(str(db))) as api:
    # 1) Sin session_id: la API crea una sesión y LA DEVUELVE
    r1 = api.post("/chat", json={"message": "Resume las notas de Luis."}).json()
    sid = r1["session_id"]
    print("1) sesión nueva:", sid, "|", r1["status"], "|", r1["answer"][:60])

    # 2) Con el mismo session_id: continúa la conversación (el modelo de juguete solo "recuerda" lo que hay en el historial)
    r2 = api.post("/chat", json={"message": "¿De quién hablábamos?", "session_id": sid}).json()
    print("2) mismo id     :", r2["answer"])

    # 3) Sin session_id: otra conversación, que no ve la anterior
    r3 = api.post("/chat", json={"message": "¿De quién hablábamos?"}).json()
    print("3) otra sesión  :", r3["answer"])

    # 4) Historial legible
    print("4) historial    :", [(m["role"], m["content"][:25]) for m in api.get(f"/sessions/{sid}").json()][:4], "...")
    print("   sesión inexistente →", api.get("/sessions/no-existe").status_code)

    # 5) Una escritura NO se ejecuta: la API responde awaiting_approval con la acción propuesta
    pend = api.post("/chat", json={"message": "Agrega una nota a Ana: 'Enviar la propuesta el lunes'", "session_id": "escritura-1"}).json()
    print("5) pide aprobar :", pend["status"], pend["pending_action"], "| escrituras:", len(crm_server.ESCRITURAS))

print("\n⏹️ La API se detuvo (salimos del `with`). El archivo sigue en disco:", db.name, f"({db.stat().st_size} bytes)")

In [ ]:
crm_server.reiniciar()

# "Reiniciamos": una aplicación NUEVA (otro proceso del servidor MCP) sobre el MISMO archivo SQLite.
with TestClient(crear_app(str(db))) as api:
    print("6) tras reiniciar, la sesión sigue:", len(api.get(f"/sessions/{sid}").json()), "mensajes")
    print("   y recuerda:", api.post("/chat", json={"message": "¿De quién hablábamos?", "session_id": sid}).json()["answer"])

    # 7) La aprobación pendiente también sobrevivió: la resolvemos por API
    hecho = api.post("/sessions/escritura-1/approve", json={"approved": True}).json()
    print("7) aprobar      :", hecho["status"], "|", hecho["answer"], "| escrituras:", len(crm_server.ESCRITURAS))

    # 8) Borrar la sesión
    print("8) borrar       :", api.delete(f"/sessions/{sid}").status_code, "→ ahora el historial da", api.get(f"/sessions/{sid}").status_code)

**Qué observar.**

- El **`session_id`** lo crea la API y lo lleva el cliente; es el `thread_id` del agente. No hay "sesión" en memoria del proceso.
- Tras "reiniciar" (otra aplicación, mismo archivo SQLite) la conversación **y** la aprobación pendiente siguen ahí: el estado está en el checkpointer, no en el proceso.
- La escritura se **pausa** (`awaiting_approval`) y se resuelve con otra llamada: es lo mismo que haría una interfaz con dos botones.

## Quién consume la API

| Cliente | Para qué |
| --- | --- |
| **Bruno** | Cliente HTTP cuyas colecciones son archivos de texto (`.bru`) en el repositorio: 13 peticiones numeradas con aserciones; un script guarda el `session_id` en una variable y las siguientes lo usan. Con `bru run` (CLI) corre sin interfaz |
| **Streamlit** (`ui.py`) | Un chat sencillo que llama a la API por HTTP (no importa el agente): muestra el `session_id`, las herramientas y los tokens, y los botones Aprobar y Rechazar |
| **`/docs`** | La documentación interactiva que FastAPI genera sola a partir de los modelos Pydantic |

> 💬 **Para pensar:** si pones dos réplicas de la API detrás de un balanceador, ¿qué necesitas para que cualquiera de ellas continúe una sesión? (Un checkpointer compartido —por ejemplo Postgres— en lugar de un archivo SQLite local; la idea es la misma.)
>
> ✍️ **Ejercicio 7.** Agrega al `crear_app` de la celda un endpoint `GET /sessions/{id}/pending` que devuelva solo la acción pendiente (usa `estado.interrupts`). Pruébalo con `TestClient`.

---
# Parte 11 · La práctica: `langchain-mcp-advanced-tooling`

La práctica vive en el repositorio **`ai-agents-playground`**, carpeta **`langchain-mcp-advanced-tooling`**. Es el servidor de la Parte 5 contra **HubSpot** (real o simulado) con las piezas separadas y probadas:

| Pieza | Qué hace |
| --- | --- |
| `mcp_server.py` | El servidor MCP de **HubSpot** (FastMCP): 3 tools, 1 recurso con plantilla y 1 prompt; stdio o Streamable HTTP |
| `hubspot_client.py` | CRM **simulado** (`HUBSPOT_MODE=fake`, por defecto) o **HubSpot real** (`live`, con el token en `.env`) |
| `agent.py`, `main.py` | El agente con `create_agent` + `langchain-mcp-adapters` + aprobación humana antes de `add_note` |
| `api.py`, `ui.py` | El agente como **API FastAPI** con sesiones en SQLite y una interfaz **Streamlit** |
| `bruno/` | Colección **Bruno**: 13 peticiones con aserciones que recorren la API paso a paso |
| `runtime.py` | Conexión en memoria, stdio, HTTP o **dos servidores a la vez** |
| `mini_client.py` | Cliente mínimo para explorar y depurar cualquier servidor |
| `.mcp.json` | Registro del servidor en Claude Code |
| `Dockerfile`, `docker-compose.yml` | Tres servicios con healthcheck: servidor MCP por HTTP, API y UI |
| `tests/` | 38 pruebas **sin red** (modelo guionado, HubSpot simulado, `TestClient` y `AppTest`) |

```bash
cd ai-agents-playground/langchain-mcp-advanced-tooling
uv sync
uv run pytest                                                              # pruebas sin red
uv run python main.py --trace "Resume las notas de Luis."                   # agente + servidor por stdio
uv run python main.py --trace --auto approve "Agrega una nota a Ana: 'Enviar la propuesta el lunes'"
uv run python main.py --multi --trace "¿Qué hora es en America/Guatemala?"  # dos servidores MCP
uv run python mini_client.py -- uv run python mcp_server.py                 # explorar el servidor
task api                                                                    # la API en http://127.0.0.1:8001/docs
task ui                                                                     # la interfaz Streamlit (en otra terminal)
task bruno                                                                  # la colección Bruno contra la API
task docker:up                                                              # MCP + API + UI en contenedores
```

> 💰 **Costo.** En modo simulado (`HUBSPOT_MODE=fake`, `LLM_PROVIDER=fake`): **US$ 0**. HubSpot tiene plan gratuito; con un modelo de pago, cada pregunta son 2 a 4 llamadas (menos de un centavo con un modelo económico).

> ⚠️ **HubSpot real.** Necesita una **Service Key** (el reemplazo recomendado de las apps privadas heredadas, en beta pública desde febrero de 2026) con los permisos `crm.objects.contacts.read` y, para `add_note`, `crm.objects.contacts.write`. Usa una **cuenta de pruebas**. Este modo no se ha probado contra una cuenta real: ver el README de la práctica.

---
# Parte 12 · Cierre

## Lo que aprendiste

1. **MCP** estandariza cómo una aplicación de IA descubre y usa herramientas y datos externos: resuelve el problema **N×M** con *host / cliente / servidor* y mensajes JSON-RPC.
2. Un servidor ofrece **tools** (las decide el modelo), **resources** (los carga la aplicación, con URI fija o plantilla) y **prompts** (los elige la persona); cada una con su pareja de métodos `list` + `call/read/get`.
3. Los transportes son **stdio** (local; `stdout` es del protocolo, los logs van a `stderr`) y **Streamable HTTP** (remoto).
4. El ciclo de vida: `initialize` → descubrimiento → uso → cierre (la revisión `2026-07-28` elimina `initialize`; el SDK actual aún lo usa).
5. Con `FastMCP`, un servidor son **funciones decoradas**; los tipos y el docstring forman el esquema que lee el modelo, y el SDK valida los argumentos antes de llamar a tu función. Las **anotaciones** describen la herramienta.
6. `langchain-mcp-adapters` convierte las tools MCP en `@tool`; el agente es el de siempre, y `HumanInTheLoopMiddleware` pide aprobación antes de escribir.
7. `MultiServerMCPClient` conecta **varios servidores** a un mismo agente; `tool_name_prefix` evita choques de nombres.
8. Un servidor se **registra** en clientes reales con un archivo (`.mcp.json`, `claude_desktop_config.json`) y se **depura** con logs a `stderr`, el cliente mínimo, el Inspector y los logs del cliente.
9. Un agente se **sirve como API**: el proceso no guarda estado; el historial vive en el *checkpointer* (SQLite) y el cliente solo lleva el `session_id` (= `thread_id`). La aprobación humana se resuelve por API (`awaiting_approval` + `POST /sessions/{id}/approve`) y la sesión sobrevive a un reinicio.

## ✅ Autoevaluación

1. ¿Qué problema resuelve MCP y por qué pasa de N×M a N+M?
2. ¿Qué diferencia hay entre *tools*, *resources* y *prompts* según **quién** decide usarlos? Da un ejemplo de cada uno.
3. ¿Cuál es la diferencia entre un recurso fijo y una plantilla de recurso?
4. ¿Por qué un `print()` en un servidor stdio puede romperlo? ¿Dónde van los logs?
5. ¿Qué mensajes componen el ciclo de vida inicial de una conexión (en `2025-11-25`)?
6. ¿Qué diferencia hay entre un error de la herramienta (`isError`) y un error del protocolo?
7. ¿Cómo se convierte un tipo de Python con restricciones en lo que ve el modelo?
8. ¿Cuándo usarías stdio y cuándo Streamable HTTP?
9. ¿Para qué sirve `tool_name_prefix` en un cliente multi-servidor?
10. ¿Qué dos archivos configuran el registro de un servidor en Claude Code y en Claude Desktop, y qué trae cada entrada stdio y HTTP?
11. Un servidor "no aparece" en el cliente. Menciona tres comprobaciones.
12. ¿Qué hace `HumanInTheLoopMiddleware` y por qué necesita un checkpointer?
13. ¿Qué relación hay entre el `session_id` de la API y el `thread_id` del checkpointer? ¿Por qué la API sigue funcionando tras reiniciarla?
14. ¿Cómo se resuelve por HTTP una escritura que necesita aprobación? ¿Qué código devuelve la API si intentas seguir chateando sin resolverla?
15. ¿Qué ventaja tiene una colección Bruno frente a probar con `curl` a mano?

## 🚀 Retos

- En la Parte 3, agrega un recurso `docs://glosario` y un prompt `traducir(texto, idioma)`.
- En la Parte 5, agrega una herramienta `update_contact_stage` con un tipo `Literal["lead", "opportunity", "customer"]` y anótala como no solo lectura.
- En la Parte 7, cambia el segundo servidor por otro servidor de referencia (por ejemplo `fetch`) y mira cómo cambian los nombres con prefijo.
- Registra `mcp_server.py` de la práctica en Claude Code (`claude mcp add ...`) y úsalo desde ahí.
- Levanta el servidor por HTTP en Docker y conéctalo con `mini_client.py --url ...`.
- En la práctica, arranca la API, ejecuta la colección Bruno, **reinicia la API** y repite la petición 04 (historial): ¿qué cambió y qué no?
- Agrega a la API un endpoint `GET /sessions/{id}/pending` que devuelva solo la acción pendiente.
- Lee la página de depuración de MCP (`modelcontextprotocol.io`, *Debugging*) y prueba el Inspector si tienes Node 22+.

## 📖 Glosario

| Término | Definición |
| --- | --- |
| **MCP** | *Model Context Protocol*: protocolo abierto para que aplicaciones de IA usen herramientas y datos externos |
| **Host / cliente / servidor** | Aplicación de IA / conector dentro de ella (1 por servidor) / programa que expone capacidades |
| **JSON-RPC 2.0** | Formato de mensajes de MCP (`method`, `params`, `id`) |
| **Tool** | Acción o consulta que decide llamar el modelo |
| **Resource** | Dato de solo lectura identificado por una URI (fija o plantilla) |
| **Prompt** | Plantilla con parámetros que elige la persona |
| **Sampling / consulta a la persona / Roots** | El servidor pide al cliente: usar su modelo / preguntar un dato a la persona / conocer las carpetas con las que trabaja |
| **stdio** | Transporte por entrada/salida estándar de un subproceso (local) |
| **Streamable HTTP** | Transporte HTTP de un endpoint, con POST y SSE opcional (remoto) |
| **`initialize`** | Primer intercambio: versión y capacidades (desaparece en la revisión `2026-07-28`) |
| **FastMCP** | API de alto nivel del SDK de Python para escribir servidores con decoradores |
| **Anotaciones de herramienta** | Metadatos (`readOnlyHint`, `destructiveHint`, `idempotentHint`, `openWorldHint`) que describen la herramienta |
| **`isError`** | Marca de un resultado de herramienta que falló; el modelo puede leerlo |
| **`MultiServerMCPClient`** | Cliente de `langchain-mcp-adapters` que conecta varios servidores a un agente |
| **`.mcp.json`** | Archivo de configuración de servidores MCP de un proyecto (Claude Code) |
| **MCP Inspector** | Interfaz web oficial para probar y depurar servidores MCP |
| **Human-in-the-loop** | Una persona aprueba o rechaza una acción antes de ejecutarse |
| **FastAPI** | Framework web de Python con validación Pydantic y documentación automática en `/docs` |
| **`session_id` / `thread_id`** | Identificador de una conversación: el cliente lo envía; el checkpointer lo usa para guardar y recuperar el historial |
| **Checkpointer (SQLite)** | Almacén persistente del estado del agente; permite reanudar una sesión tras reiniciar la API |
| **`awaiting_approval`** | Estado de una sesión pausada esperando que una persona apruebe o rechace una escritura |
| **Streamlit** | Biblioteca para construir interfaces web sencillas en Python |
| **Bruno** | Cliente HTTP cuyas colecciones son archivos de texto (`.bru`), con aserciones y variables; tiene CLI |
| **Service Key (HubSpot)** | Credencial de cuenta para integraciones de datos; reemplaza a las apps privadas heredadas |

## ➡️ Siguiente

El jueves siguiente: **LangGraph**. Construirás un agente con todo lo visto (RAG, memoria, herramientas, MCP) como un **grafo explícito**, con aprobación humana antes de escribir, contra un sistema externo distinto (**Trello**). La seguridad de MCP se verá en el Módulo 2.